# Biohub window-3 probe — unit-sliced training (STRATEGY 49)

**One-time setup:** add a Colab Secret named `KAGGLE_JSON` containing the full contents of your `kaggle.json` (Kaggle → Settings → API → Create New Token). Enable notebook access for the secret.

**Every session:** Runtime → Change runtime type → **A100** (L4 works, ~40% slower per unit). Then *Run all*. The session downloads data (~25 min), resumes from the last checkpoint, trains until the time budget, and uploads the checkpoint back to Kaggle. Re-run whenever you have compute units — progress accumulates automatically.

In [ ]:
#@title Session parameters
MAX_MINUTES = 165      #@param {type:"integer"}  — leave ~30 min of session for data+upload
BATCH_SIZE  = 8        #@param {type:"integer"}  — drop to 4 if OOM (L4/T4)
EPOCHS      = 60       #@param {type:"integer"}  — total target; sessions accumulate toward it
import torch, subprocess
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
print("GPU:", gpu)
assert gpu != "NONE", "Set Runtime -> A100/L4 first"


In [ ]:
#@title Dependencies (~2 min)
%pip -q install "zarr>=3.0.10,<4" blosc2 "numcodecs>=0.13,<0.16" "geff>=1.1.3.1.1" "geff-spec<1.2" tracksdata "polars>=1.36" dask tqdm rustworkx "sqlalchemy>=2" 2>&1 | tail -1
import zarr; print("zarr", zarr.__version__)


In [ ]:
#@title Kaggle auth (from Colab Secret KAGGLE_JSON)
import os, pathlib
from google.colab import userdata
kj = userdata.get('KAGGLE_JSON')
p = pathlib.Path('/root/.kaggle'); p.mkdir(exist_ok=True)
(p / 'kaggle.json').write_text(kj)
os.chmod(p / 'kaggle.json', 0o600)
!pip -q install -U kaggle 2>&1 | tail -1
!kaggle competitions list -s biohub | head -3


In [ ]:
#@title Competition data (~25 min first run; cached on the session disk)
import os
if not os.path.exists('/content/data/train'):
    !mkdir -p /content/data
    !kaggle competitions download -c biohub-cell-tracking-during-development -p /content/data
    print("unzipping (this is the slow part)...")
    !cd /content/data && unzip -qo biohub-cell-tracking-during-development.zip && rm biohub-cell-tracking-during-development.zip
n = len([d for d in os.listdir('/content/data/train') if d.endswith('.zarr')])
print(f"train mount: {n} zarr stores")
assert n >= 190


In [ ]:
#@title Trainer: support pack + resume/budget patch + splits
import os, json, pathlib
if not os.path.exists('/content/repo'):
    !kaggle datasets download pilkwang/biohub-tracking-support-pack-50ep-v1 -p /content/sp --unzip
    !cp -r /content/sp/repo /content/repo

PATCHER = r"""#!/usr/bin/env python3
\"\"\"Patch the full-recipe trainer for unit-sliced Colab training (STRATEGY 48/49).

The support-pack `train_unet_transformer.py` saves only `edge_predictor_best.pth`
(flat model weights) — a session that dies loses every epoch since the last
best. For Colab Pro's unit-sliced reality (sessions of a few hours, spread over
days), the trainer needs exactly what we built for ctrack in row 40:

  * `checkpoint_last.pth` — model + optimizer + epoch + best_score, written
    ATOMICALLY every epoch;
  * `--resume auto` — continue from it if present;
  * `--max-minutes` — refuse to start an epoch that can't finish inside the
    session budget, so the session ends cleanly with the checkpoint uploadable.

Applied with exact-count asserts (a patch that silently fails to apply looks
exactly like success). Shared by the local M1 smoke and the Colab notebook so
both run the same code.

    python scripts/local/patch_fullrecipe_trainer.py <path-to-train_unet_transformer.py>
\"\"\"

from __future__ import annotations

import sys
from pathlib import Path

ANCHOR_ARGS = '''    parser.add_argument("--single-gpu", dest="data_parallel", action="store_false",'''
PATCH_ARGS = '''    parser.add_argument("--resume", type=str, default="auto",
                        help="'auto' resumes from checkpoint_last.pth in the output dir if "
                             "present; 'none' starts fresh")
    parser.add_argument("--max-minutes", type=float, default=None,
                        help="stop before starting an epoch that would exceed this budget")
    parser.add_argument("--single-gpu", dest="data_parallel", action="store_false",'''

ANCHOR_SIG = '''    max_iters: int | None = None,
    debug_video: Path | None = None,'''
PATCH_SIG = '''    max_iters: int | None = None,
    resume: str = "auto",
    max_minutes: float | None = None,
    debug_video: Path | None = None,'''

ANCHOR_CALL = '''            max_iters=args.max_iters,'''
PATCH_CALL = '''            max_iters=args.max_iters,
            resume=args.resume,
            max_minutes=args.max_minutes,'''

ANCHOR_LOOP = '''    best_score = 0.0
    save_path = output_dir / "edge_predictor_best.pth"
    pbar = tqdm(range(n_epochs), desc="Training", disable=False)
    print(f"Detection loss: weight={det_loss_weight}, neg_weight={det_neg_weight}", flush=True)

    for epoch in pbar:
        t0 = time.monotonic()'''
PATCH_LOOP = '''    best_score = 0.0
    save_path = output_dir / "edge_predictor_best.pth"
    last_path = output_dir / "checkpoint_last.pth"

    # ---- resume (unit-sliced Colab sessions; STRATEGY 48) ----
    start_epoch = 0
    if resume == "auto" and last_path.exists():
        _ck = torch.load(last_path, map_location=device, weights_only=False)
        _state = _ck["model_state_dict"]
        if isinstance(model.unet, nn.DataParallel):
            _state = {(k.replace("unet.", "unet.module.", 1)
                       if k.startswith("unet.") and not k.startswith("unet.module.") else k): v
                      for k, v in _state.items()}
        model.load_state_dict(_state)
        optimizer.load_state_dict(_ck["optimizer_state_dict"])
        start_epoch = int(_ck["epoch"]) + 1
        best_score = float(_ck.get("best_score", 0.0))
        print(f"RESUMED from {last_path.name}: epoch {start_epoch} "
              f"(best {best_score:.4f}, {len(_ck['model_state_dict'])} tensors)", flush=True)
    elif resume == "auto":
        print("fresh run (no checkpoint_last.pth)", flush=True)

    _run_t0 = time.monotonic()
    _est_epoch_s = None
    pbar = tqdm(range(start_epoch, n_epochs), desc="Training", disable=False)
    print(f"Detection loss: weight={det_loss_weight}, neg_weight={det_neg_weight}", flush=True)

    for epoch in pbar:
        if max_minutes is not None and _est_epoch_s is not None:
            _elapsed_min = (time.monotonic() - _run_t0) / 60.0
            if _elapsed_min + _est_epoch_s / 60.0 > max_minutes:
                print(f"BUDGET STOP: {_elapsed_min:.1f} min elapsed, next epoch "
                      f"~{_est_epoch_s/60:.1f} min exceeds {max_minutes:.0f} min. "
                      f"Resumable at epoch {epoch}.", flush=True)
                break
        t0 = time.monotonic()'''

ANCHOR_SAVE = '''        marker = "*" if is_best else " "'''
PATCH_SAVE = '''        # ---- atomic resumable checkpoint, every epoch ----
        _est_epoch_s = train_time + test_time
        _tmp = last_path.with_suffix(".tmp")
        torch.save({
            "epoch": epoch,
            "best_score": best_score,
            "model_state_dict": {k.replace("unet.module.", "unet.", 1): v
                                 for k, v in model.state_dict().items()},
            "optimizer_state_dict": optimizer.state_dict(),
        }, _tmp)
        _tmp.replace(last_path)

        marker = "*" if is_best else " "'''


def patch(path: Path) -> None:
    s = path.read_text()
    if "BUDGET STOP" in s:
        print("already patched")
        return
    for name, old, new in [("args", ANCHOR_ARGS, PATCH_ARGS),
                           ("sig", ANCHOR_SIG, PATCH_SIG),
                           ("call", ANCHOR_CALL, PATCH_CALL),
                           ("loop", ANCHOR_LOOP, PATCH_LOOP),
                           ("save", ANCHOR_SAVE, PATCH_SAVE)]:
        n = s.count(old)
        assert n == 1, f"anchor {name!r} matched {n} times, expected 1"
        s = s.replace(old, new, 1)
    compile(s, str(path), "exec")
    path.write_text(s)
    print(f"patched {path} (5/5 anchors, compile OK)")


if __name__ == "__main__":
    patch(Path(sys.argv[1]))
"""
pathlib.Path('/content/patcher.py').write_text(PATCHER)
!cd /content && python patcher.py /content/repo/scripts/train_unet_transformer.py

SPLITS = r"""[
 {
  "train": [
   "44b6_0db75fae.zarr",
   "44b6_12dfb391.zarr",
   "44b6_1574802b.zarr",
   "44b6_18ced818.zarr",
   "44b6_1d530831.zarr",
   "44b6_24264f12.zarr",
   "44b6_267148e4.zarr",
   "44b6_2a2eff9f.zarr",
   "44b6_341df25f.zarr",
   "44b6_3bb3690f.zarr",
   "44b6_53f95252.zarr",
   "44b6_551a5dba.zarr",
   "44b6_5f15d135.zarr",
   "44b6_66f9292d.zarr",
   "44b6_706092f0.zarr",
   "44b6_71a4179f.zarr",
   "44b6_7a302da0.zarr",
   "44b6_7e557709.zarr",
   "44b6_808952d6.zarr",
   "44b6_90724892.zarr",
   "44b6_95029e92.zarr",
   "44b6_9be80b04.zarr",
   "44b6_a21120c2.zarr",
   "44b6_b2c44266.zarr",
   "44b6_c771cb04.zarr",
   "44b6_cf2536e8.zarr",
   "44b6_d29c9ab2.zarr",
   "44b6_d2f34f90.zarr",
   "44b6_db3c847b.zarr",
   "44b6_ddf577ad.zarr",
   "44b6_e28840c6.zarr",
   "44b6_e29f0176.zarr",
   "44b6_f28707c6.zarr",
   "6bba_0c7fa718.zarr",
   "6bba_13f19531.zarr",
   "6bba_1d0d8384.zarr",
   "6bba_1ebfb80d.zarr",
   "6bba_1f58c2f6.zarr",
   "6bba_20852818.zarr",
   "6bba_2312ac41.zarr",
   "6bba_23af9eeb.zarr",
   "6bba_268e1230.zarr",
   "6bba_312f0dc3.zarr",
   "6bba_32db13fc.zarr",
   "6bba_372c8cb8.zarr",
   "6bba_3abfe10a.zarr",
   "6bba_3c5691b6.zarr",
   "6bba_3fda6b25.zarr",
   "6bba_43fea39d.zarr",
   "6bba_4aa14c89.zarr",
   "6bba_5a4d9360.zarr",
   "6bba_5b28472a.zarr",
   "6bba_5dfe9ad1.zarr",
   "6bba_67ebd073.zarr",
   "6bba_6ca87370.zarr",
   "6bba_6db0e9b4.zarr",
   "6bba_6feb10f0.zarr",
   "6bba_6feeb0b1.zarr",
   "6bba_705ec2c9.zarr",
   "6bba_74686d6a.zarr",
   "6bba_767a1e17.zarr",
   "6bba_76db78c1.zarr",
   "6bba_78a7bd97.zarr",
   "6bba_7f87b3d8.zarr",
   "6bba_8b7818bf.zarr",
   "6bba_907271db.zarr",
   "6bba_91951b3a.zarr",
   "6bba_9a41d029.zarr",
   "6bba_9e23430b.zarr",
   "6bba_a90a0b9c.zarr",
   "6bba_acd782a8.zarr",
   "6bba_aeee7805.zarr",
   "6bba_af149c94.zarr",
   "6bba_afb141ff.zarr",
   "6bba_b1ae37b9.zarr",
   "6bba_b693381b.zarr",
   "6bba_bb9f20c3.zarr",
   "6bba_c27cba08.zarr",
   "6bba_c73a1d11.zarr",
   "6bba_cdcfe533.zarr",
   "6bba_cf35214c.zarr",
   "6bba_cff5865f.zarr",
   "6bba_d0fc38b5.zarr",
   "6bba_d1acb6ff.zarr",
   "6bba_d2b9fc0c.zarr",
   "6bba_d3da753b.zarr",
   "6bba_d5eae175.zarr",
   "6bba_e5e44988.zarr",
   "6bba_ebdf3b34.zarr",
   "6bba_ebff6e76.zarr",
   "6bba_f17befbc.zarr",
   "6bba_f1fde7e0.zarr",
   "6bba_f20478e9.zarr",
   "6bba_f4ae811c.zarr",
   "6bba_f8ffd5e7.zarr",
   "6bba_fc516dc6.zarr"
  ],
  "test": [
   "44b6_5740d24b.zarr",
   "44b6_8cc6506c.zarr",
   "44b6_8f9ecab4.zarr",
   "6bba_062c8d37.zarr",
   "6bba_207c6aaf.zarr",
   "6bba_2540cd90.zarr",
   "6bba_3a1849c2.zarr",
   "6bba_784a78c9.zarr",
   "6bba_971fa5e0.zarr",
   "6bba_a5e926bb.zarr",
   "6bba_ab78413d.zarr",
   "6bba_fc83837d.zarr"
  ],
  "_train_full": [
   "44b6_0113de3b.zarr",
   "44b6_0b24845f.zarr",
   "44b6_0c582fdc.zarr",
   "44b6_0db75fae.zarr",
   "44b6_12dfb391.zarr",
   "44b6_144b256d.zarr",
   "44b6_1574802b.zarr",
   "44b6_18ced818.zarr",
   "44b6_1d530831.zarr",
   "44b6_24264f12.zarr",
   "44b6_267148e4.zarr",
   "44b6_2a2eff9f.zarr",
   "44b6_33b596bf.zarr",
   "44b6_341df25f.zarr",
   "44b6_3a861e03.zarr",
   "44b6_3bb3690f.zarr",
   "44b6_40c45f5a.zarr",
   "44b6_415c0a3a.zarr",
   "44b6_53f95252.zarr",
   "44b6_551a5dba.zarr",
   "44b6_587a1e22.zarr",
   "44b6_5f15d135.zarr",
   "44b6_668e0cc7.zarr",
   "44b6_66f9292d.zarr",
   "44b6_706092f0.zarr",
   "44b6_71a4179f.zarr",
   "44b6_74d0c52e.zarr",
   "44b6_7a302da0.zarr",
   "44b6_7e557709.zarr",
   "44b6_808952d6.zarr",
   "44b6_81c256f0.zarr",
   "44b6_87bba6c4.zarr",
   "44b6_8f5ab931.zarr",
   "44b6_90724892.zarr",
   "44b6_949adeb1.zarr",
   "44b6_95029e92.zarr",
   "44b6_97725144.zarr",
   "44b6_996155de.zarr",
   "44b6_9be80b04.zarr",
   "44b6_9bfa6a0a.zarr",
   "44b6_a21120c2.zarr",
   "44b6_aaf8b0ea.zarr",
   "44b6_abf82518.zarr",
   "44b6_b2c44266.zarr",
   "44b6_c15fded2.zarr",
   "44b6_c50204e0.zarr",
   "44b6_c771cb04.zarr",
   "44b6_c8e2a523.zarr",
   "44b6_cf2536e8.zarr",
   "44b6_cf8fed6b.zarr",
   "44b6_d29c9ab2.zarr",
   "44b6_d2f34f90.zarr",
   "44b6_d5e7d891.zarr",
   "44b6_d754aa59.zarr",
   "44b6_d78e09d9.zarr",
   "44b6_db3c847b.zarr",
   "44b6_ddf577ad.zarr",
   "44b6_deabac95.zarr",
   "44b6_e28840c6.zarr",
   "44b6_e29f0176.zarr",
   "44b6_e31261b4.zarr",
   "44b6_e35b117d.zarr",
   "44b6_e57ff5c6.zarr",
   "44b6_f28707c6.zarr",
   "6bba_05b6850b.zarr",
   "6bba_05db0fb1.zarr",
   "6bba_07477033.zarr",
   "6bba_07e24132.zarr",
   "6bba_085bf656.zarr",
   "6bba_09961292.zarr",
   "6bba_0c7fa718.zarr",
   "6bba_0e7c0d07.zarr",
   "6bba_12665c0e.zarr",
   "6bba_13f19531.zarr",
   "6bba_15403b9a.zarr",
   "6bba_1d0d8384.zarr",
   "6bba_1ebfb80d.zarr",
   "6bba_1f58c2f6.zarr",
   "6bba_20852818.zarr",
   "6bba_2312ac41.zarr",
   "6bba_23af9eeb.zarr",
   "6bba_2646afc7.zarr",
   "6bba_268e1230.zarr",
   "6bba_2819ca14.zarr",
   "6bba_283bf9f1.zarr",
   "6bba_312f0dc3.zarr",
   "6bba_32db13fc.zarr",
   "6bba_337b1b3a.zarr",
   "6bba_372c8cb8.zarr",
   "6bba_3abfe10a.zarr",
   "6bba_3c5691b6.zarr",
   "6bba_3db54e20.zarr",
   "6bba_3fda6b25.zarr",
   "6bba_43fea39d.zarr",
   "6bba_474be664.zarr",
   "6bba_48816121.zarr",
   "6bba_4aa14c89.zarr",
   "6bba_4f99ce20.zarr",
   "6bba_4ffd3da3.zarr",
   "6bba_55b7eebe.zarr",
   "6bba_55c70843.zarr",
   "6bba_57b7cc1e.zarr",
   "6bba_5a4d9360.zarr",
   "6bba_5b28472a.zarr",
   "6bba_5c039895.zarr",
   "6bba_5c824876.zarr",
   "6bba_5dfe9ad1.zarr",
   "6bba_5f89039d.zarr",
   "6bba_61dd1e0d.zarr",
   "6bba_61ecbe65.zarr",
   "6bba_6321a359.zarr",
   "6bba_6479435d.zarr",
   "6bba_67ebd073.zarr",
   "6bba_6ca87370.zarr",
   "6bba_6db0e9b4.zarr",
   "6bba_6feb10f0.zarr",
   "6bba_6feeb0b1.zarr",
   "6bba_705ec2c9.zarr",
   "6bba_718b21f9.zarr",
   "6bba_74686d6a.zarr",
   "6bba_767a1e17.zarr",
   "6bba_76db78c1.zarr",
   "6bba_786893ac.zarr",
   "6bba_789f8168.zarr",
   "6bba_78a7bd97.zarr",
   "6bba_7af54fde.zarr",
   "6bba_7d3058ae.zarr",
   "6bba_7f87b3d8.zarr",
   "6bba_80d12824.zarr",
   "6bba_825bd1c6.zarr",
   "6bba_87289e13.zarr",
   "6bba_8b7818bf.zarr",
   "6bba_907271db.zarr",
   "6bba_91951b3a.zarr",
   "6bba_96833384.zarr",
   "6bba_969618f6.zarr",
   "6bba_9a41d029.zarr",
   "6bba_9e23430b.zarr",
   "6bba_a90a0b9c.zarr",
   "6bba_acd782a8.zarr",
   "6bba_ae82a791.zarr",
   "6bba_aeee7805.zarr",
   "6bba_af149c94.zarr",
   "6bba_afb141ff.zarr",
   "6bba_b1ae37b9.zarr",
   "6bba_b204cac7.zarr",
   "6bba_b693381b.zarr",
   "6bba_bb9f20c3.zarr",
   "6bba_bbb708ca.zarr",
   "6bba_c27cba08.zarr",
   "6bba_c328f2fd.zarr",
   "6bba_c73a1d11.zarr",
   "6bba_cdcfe533.zarr",
   "6bba_cf35214c.zarr",
   "6bba_cff5865f.zarr",
   "6bba_d0fc38b5.zarr",
   "6bba_d1acb6ff.zarr",
   "6bba_d2b9fc0c.zarr",
   "6bba_d3da753b.zarr",
   "6bba_d5eae175.zarr",
   "6bba_d6ecebbb.zarr",
   "6bba_df673a83.zarr",
   "6bba_e16ffc58.zarr",
   "6bba_e5e44988.zarr",
   "6bba_ebdf3b34.zarr",
   "6bba_ebff6e76.zarr",
   "6bba_ed9377fd.zarr",
   "6bba_edf14583.zarr",
   "6bba_eebc57a5.zarr",
   "6bba_ef7b4f7e.zarr",
   "6bba_f17befbc.zarr",
   "6bba_f1fde7e0.zarr",
   "6bba_f20478e9.zarr",
   "6bba_f4ae811c.zarr",
   "6bba_f8ffd5e7.zarr",
   "6bba_fbc898dc.zarr",
   "6bba_fc516dc6.zarr",
   "6bba_fc5f39dc.zarr",
   "6bba_fe670320.zarr"
  ],
  "_golden_holdout_never_used": [
   "44b6_2f31fc2f.zarr",
   "44b6_a2bb48bb.zarr",
   "44b6_c96cfa10.zarr",
   "44b6_eb2880fc.zarr",
   "6bba_7b5d3b2c.zarr",
   "6bba_b329af44.zarr",
   "6bba_d82a4fc6.zarr",
   "6bba_debd7bfa.zarr"
  ]
 }
]"""
pathlib.Path('/content/w3_probe_splits.json').write_text(SPLITS)
sp = json.loads(SPLITS)[0]
print(f"splits: train {len(sp['train'])} | selection {len(sp['test'])} | golden quarantined {len(sp['_golden_holdout_never_used'])}")


In [ ]:
#@title Resume from the checkpoint store (if any)
import os, glob, shutil
out_dir = '/content/repo/weights/w3probe/split_0'
os.makedirs(out_dir, exist_ok=True)
!kaggle datasets download maruanbottoni/biohub-w3-ckpt -p /content/ckpt --unzip || true
ck = glob.glob('/content/ckpt/**/checkpoint_last.pth', recursive=True)
if ck:
    shutil.copy(ck[0], f'{out_dir}/checkpoint_last.pth')
    print('RESUME SEED:', ck[0])
    for extra in ('edge_predictor_best.pth', 'config.json'):
        e = glob.glob(f'/content/ckpt/**/{extra}', recursive=True)
        if e: shutil.copy(e[0], f'{out_dir}/{extra}')
else:
    print('fresh run — no checkpoint in the store yet')


In [ ]:
#@title TRAIN (window-3, resumable, budget-stopped)
import subprocess, sys, os
env = {**os.environ, 'PYTHONPATH': 'src'}
cmd = [sys.executable, 'scripts/train_unet_transformer.py',
       '--data-dir', '/content/data/train',
       '--splits', '/content/w3_probe_splits.json', '--split', '0',
       '--method', 'w3probe', '--window-size', '3',
       '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
       '--num-workers', '4', '--single-gpu',
       '--max-minutes', str(MAX_MINUTES)]
print(' '.join(cmd))
r = subprocess.run(cmd, cwd='/content/repo', env=env)
print('trainer exit:', r.returncode)


In [ ]:
#@title Upload checkpoint back to the store (ALWAYS run this cell)
import json, os, shutil, pathlib, glob
st = pathlib.Path('/content/w3ckpt_staging'); st.mkdir(exist_ok=True)
src = '/content/repo/weights/w3probe/split_0'
n = 0
for f in ('checkpoint_last.pth', 'edge_predictor_best.pth', 'config.json'):
    if os.path.exists(f'{src}/{f}'):
        shutil.copy(f'{src}/{f}', st / f); n += 1
assert n >= 1, 'nothing to upload — did training run?'
(st / 'dataset-metadata.json').write_text(json.dumps({
    'id': 'maruanbottoni/biohub-w3-ckpt', 'title': 'biohub-w3-ckpt',
    'licenses': [{'name': 'CC0-1.0'}]}))
import torch
ck = torch.load(st / 'checkpoint_last.pth', map_location='cpu', weights_only=False)
msg = f"epoch {ck['epoch']} best {ck.get('best_score', 0):.4f} ({os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()})"
!kaggle datasets version -p /content/w3ckpt_staging -m "{msg}"
print('uploaded:', msg)
